# 快速入门
## 1. 构建一个基本代理
创建一个可以回答问题和调用工具的简单代理。代理使用基于Ollama本地部署的qwen3:4b作为其语言模型，一个基本天气功能作为工具，以及一个简单提示来指导其行为。

In [13]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:1.7b")

# Define a simple weather function
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

# Create the agent
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='41f37dd0-1c5f-4e3d-92c3-a7afb61c3883'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:1.7b', 'created_at': '2026-01-13T06:40:52.2549178Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10092333200, 'load_duration': 82742200, 'prompt_eval_count': 148, 'prompt_eval_duration': 3211182700, 'eval_count': 120, 'eval_duration': 6772353000, 'logprobs': None, 'model_name': 'qwen3:1.7b', 'model_provider': 'ollama'}, id='lc_run--019bb615-d2b1-7993-a7f4-2c7026e9c1f4-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'sf'}, 'id': '397187b7-bf19-432b-9eb0-1943a51cc63b', 'type': 'tool_call'}], usage_metadata={'input_tokens': 148, 'output_tokens': 120, 'total_tokens': 268}),
  ToolMessage(content="It's always sunny in sf!", name='get_weather', id='a00cce2e-2319-4f0a-bb3c-ba5a16fc2d0d', tool_call_id='397187b7-bf19-432b-9eb0-1943a51cc63b'),
  A

## 2. 构建一个真实世界的代理
构建一个实用的天气预报代理，演示关键的生产概念

1. 详细的系统提示以获得更好的代理行为
2. 创建与外部数据集成的工具
3. 模型配置以获得一致的响应
4. 结构化输出以获得可预测的结果
5. 对话记忆以进行类似聊天的交互
6. 创建并运行代理以创建功能齐全的代理
### 2.1 定义系统提示
系统提示定义了代理的角色和行为。保持其具体和可操作

In [11]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

### 2.2 创建工具

**工具**允许模型通过调用定义的函数与外部系统交互。工具可以依赖于**运行时上下文**，也可以与**代理**记忆交互。

请注意下面 `get_user_location` 工具如何使用运行时上下文：

In [14]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool # langchain的工具装饰器，将函数注册为代理可调用的工具
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass # 类似Java中的Lombok的@Data
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

### 2.3 配置模型
为用例设置具有正确参数的语言模型

In [15]:
# from langchain.chat_models import init_chat_model

# model = init_chat_model(
#     "claude-sonnet-4-5-20250929",
#     temperature=0.5,
#     timeout=10,
#     max_tokens=1000
# )

from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0.5, # 0.0~1.0 较低值产生更确定性的输出，较高值增加创造性
    timeout=10, # 超时时间
    max_tokens=1000 # 单词响应的最大 token 数量
)

### 2.4 定义响应格式

如果需要代理响应匹配特定模式，可以选择定义结构化响应格式。

In [16]:
from dataclasses import dataclass

# We use a dataclass here, but Pydantic models are also supported.
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

### 2.5 添加记忆
为代理添加记忆，以在交互过程中保持状态。这允许代理记住之前的对话和上下文。

In [17]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

### 2.6 创建并运行代理
现在，用所有组件组装您的代理并运行它！

In [18]:
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context, # 指定运行时上下文的数据结构
    response_format=ResponseFormat, # 指定响应的数据结构
    checkpointer=checkpointer # 添加记忆
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

ResponseFormat(punny_response="It's always sunny in Florida! ☀️", weather_conditions='sun: ☀️, temperature: 75°F')
ResponseFormat(punny_response="It's always sunny in Florida! ☀️", weather_conditions='sun: ☀️, temperature: 75°F')
